# Synthetic Data Generation


In [1]:
# ============================================================
# IMPORT DEPENDENCIES
# ============================================================

import pandas as pd
import random


random.seed(42)
random_state = 42

print("Setup complete.")


Setup complete.


### These are the 12 app actions our classifier must recognize.

In [2]:
intents = [
    "check_balance", "send_money", "buy_airtime", "buy_data",
    "call_contact", "open_app", "check_weather", "set_reminder",
    "play_music", "stop_action", "greeting", "help_request",
]
print(f"{len(intents)} intents:", intents)


12 intents: ['check_balance', 'send_money', 'buy_airtime', 'buy_data', 'call_contact', 'open_app', 'check_weather', 'set_reminder', 'play_music', 'stop_action', 'greeting', 'help_request']


## 2. Generate the Synthetic Labeled Dataset

Real labeled Pidgin/English command datasets are scarce, so I generate
realistic command variations from templates that mix formal English,
Nigerian Pidgin, and messy real-world phrasing (typos, filler words).

In [3]:
# Slot values used to fill in template placeholders
names = ["mama", "papa", "my broda", "my sista", "Chidi", "Ngozi", "Tunde",
         "Amaka", "Emeka", "Blessing", "Ibrahim", "Kemi", "Uncle Femi",
         "Aunty Bisi", "my oga", "my guy", "Fatima", "Musa", "Chinedu", "Grace"]
amounts = ["500 naira", "1000 naira", "2k", "5000 naira", "10k", "200 naira",
           "1500 naira", "3000 naira", "20k", "50 naira", "N500", "N2000"]
apps = ["camera", "whatsapp", "facebook", "gallery", "settings", "calculator",
        "phone book", "gmail", "youtube", "the browser", "instagram", "playstore"]
bills = ["light bill", "school fees", "rent", "DSTV subscription", "water bill",
         "generator fuel", "shop rent", "electricity bill", "church offering"]
times = ["by 6am", "later today", "tomorrow morning", "by 5 o'clock",
         "this evening", "before 12", "next week", "on Monday", "by weekend"]
networks = ["MTN", "Glo", "Airtel", "9mobile"]
song_or_artist = ["some music", "afrobeat", "Burna Boy", "gospel song",
                  "Wizkid", "highlife", "my playlist", "fuji music", "Davido"]

templates = {
    "check_balance": [
        "how much I get for my account", "check my balance",
        "abeg check my account balance", "wetin be my balance",
        "I wan know how much dey my account", "show me my account balance",
        "what is my current balance", "how much money I get now",
        "balance check abeg", "confirm my balance make I see",
    ],
    "send_money": [
        "send {amount} give {name}", "I wan send money to {name}",
        "transfer {amount} to {name}", "abeg send {amount} give {name} sharp sharp",
        "send money give {name} now now", "please transfer {amount} to {name}'s account",
        "I want to send {amount} to {name}", "make transfer of {amount} go {name}",
    ],
    "buy_airtime": [
        "buy {amount} airtime for me", "recharge my line with {amount}",
        "abeg buy credit give me", "top up my {network} line with {amount}",
        "I wan buy recharge card of {amount}", "load {amount} credit for my phone",
        "buy airtime {amount} on {network}",
    ],
    "buy_data": [
        "buy data for me", "I wan buy data bundle", "recharge my data with {amount}",
        "abeg buy {network} data bundle", "top up my data plan",
        "buy {amount} data on {network} for me", "load internet data for my phone",
    ],
    "call_contact": [
        "call {name}", "abeg call {name} for me", "dial {name}'s number",
        "phone {name} now", "I wan call {name}", "connect me to {name}", "ring {name}",
    ],
    "open_app": [
        "open {app}", "abeg open {app} for me", "launch {app}", "I wan use {app}",
        "start {app}", "bring up {app}", "show me {app}",
    ],
    "check_weather": [
        "how the weather be today", "wetin be the weather like",
        "is it going to rain today", "check weather for me", "abeg tell me the weather",
        "will rain fall today", "what is the weather forecast", "e go rain today?",
    ],
    "set_reminder": [
        "remind me make I pay {bill} {time}", "abeg remind me about {bill}",
        "set reminder for {bill} {time}", "I wan set alarm to remember {bill}",
        "don't make me forget {bill} {time}", "please remind me to pay {bill}",
        "set a reminder for me {time}",
    ],
    "play_music": [
        "play {song}", "abeg play {song} for me", "I wan hear {song}",
        "put on {song}", "start playing {song}", "play some music na", "gimme {song}",
    ],
    "stop_action": [
        "stop am", "cancel this one", "abeg stop", "pause everything",
        "no do am again", "stop the music", "cancel that action", "halt",
    ],
    "greeting": [
        "how far", "good morning", "how you dey", "good afternoon",
        "wetin dey happen", "hope you dey fine", "hello there", "good evening my friend",
    ],
    "help_request": [
        "help me abeg", "wetin I go do", "I no understand, help me", "abeg I need help",
        "how does this work", "I dey confuse, assist me", "please guide me", "I need assistance",
    ],
}
print("Templates defined for all", len(templates), "intents.")


Templates defined for all 12 intents.


In [4]:
filler_prefixes = ["", "", "", "abeg ", "please ", "biko ", "oga "]
filler_suffixes = ["", "", "", " abeg", " please", " o", " na", " jare"]

def typo_noise(word):
    """Simulates a realistic typing/transcription slip on one word."""
    if len(word) <= 3:
        return word
    idx = random.randint(0, len(word) - 2)
    choice = random.random()
    if choice < 0.34:
        chars = list(word)
        chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]
        return "".join(chars)
    elif choice < 0.67:
        return word[:idx] + word[idx + 1:]
    else:
        return word[:idx] + word[idx] + word[idx:]

def add_noise(sentence, typo_prob=0.12):
    """Adds filler words, random typos, and capitalization noise."""
    prefix = random.choice(filler_prefixes)
    suffix = random.choice(filler_suffixes)
    sentence = f"{prefix}{sentence}{suffix}".strip()
    words = sentence.split()
    words = [typo_noise(w) if random.random() < typo_prob else w for w in words]
    sentence = " ".join(words)
    if random.random() < 0.5 and sentence:
        sentence = sentence[0].upper() + sentence[1:]
    return " ".join(sentence.split())

def fill_template(template):
    return template.format(
        name=random.choice(names), amount=random.choice(amounts),
        app=random.choice(apps), bill=random.choice(bills),
        time=random.choice(times), network=random.choice(networks),
        song=random.choice(song_or_artist),
    )

def generate_dataset(samples_per_intent=80):
    rows = []
    for intent in intents:
        intent_templates = templates[intent]
        seen = set()
        attempts = 0
        while len(seen) < samples_per_intent and attempts < samples_per_intent * 20:
            attempts += 1
            filled = fill_template(random.choice(intent_templates))
            noisy = add_noise(filled)
            key = noisy.lower()
            if key not in seen:
                seen.add(key)
                rows.append((noisy, intent))
    random.shuffle(rows)
    return rows

dataset_rows = generate_dataset(samples_per_intent=80)
df = pd.DataFrame(dataset_rows, columns=["command", "intent"])
df.to_csv(r"C:\Users\USER\Desktop\Notebooks\voice_command_classifier\data\commands_dataset.csv", index=False)
print(f"Generated {len(df)} labeled commands.")
df.sample(10, random_state=1)


Generated 960 labeled commands.


,command,intent
241,Oga what is the weather orecast,check_weather
851,Biko how much I get for my account jare,check_balance
436,Biko load 1000 naira credit for my phone,buy_airtime
386,I wan use facebook,open_app
345,I wan use youtube please,open_app
311,Oga how much money I get now beg,check_balance
874,cancel that action are,stop_action
267,Is it going to rain today na,check_weather
35,play some music na o,play_music
78,abeg ssend money give my oga now now please,send_money
